In [33]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed

# Creating combinations of planforms

In [34]:
span_max = 4. #TODO change

lists_to_recombine = dict()

lists_to_recombine['aspect_ratio'] = [5., 10., 17., 27.]
lists_to_recombine['taper'] = [1.]
lists_to_recombine['thickness_to_chord'] = [.06, .12, .18] #NOTE not super justified
lists_to_recombine['sweep'] = [-20., 40.] #NOTE not super justified
lists_to_recombine['cl_alpha'] = [2*np.pi]
lists_to_recombine['cl_max'] = [1.]
lists_to_recombine['cm_ac'] = [-.05, 0.05] #TODO change
lists_to_recombine['cl_0'] = [0.]
lists_to_recombine['pf_type'] = ['tail', 'canard']
lists_to_recombine['pf_stable'] = [True, False]

In [35]:
ltr_keys = lists_to_recombine.keys()
ltr_values = lists_to_recombine.values()

planforms_raw = list(itt.product(*ltr_values))
print(planforms_raw)

planform_params = list()
for planform_raw in planforms_raw:
    planform_param = dict()
    for i, key in enumerate(ltr_keys):
        planform_param[key] = planform_raw[i]
    planform_params.append(planform_param)

assert len(planform_params) == 192, len(planform_params)

[(5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, -0.05, 0.0, 'tail', True), (5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, -0.05, 0.0, 'tail', False), (5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, -0.05, 0.0, 'canard', True), (5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, -0.05, 0.0, 'canard', False), (5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, 0.05, 0.0, 'tail', True), (5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, 0.05, 0.0, 'tail', False), (5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, 0.05, 0.0, 'canard', True), (5.0, 1.0, 0.06, -20.0, 6.283185307179586, 1.0, 0.05, 0.0, 'canard', False), (5.0, 1.0, 0.06, 40.0, 6.283185307179586, 1.0, -0.05, 0.0, 'tail', True), (5.0, 1.0, 0.06, 40.0, 6.283185307179586, 1.0, -0.05, 0.0, 'tail', False), (5.0, 1.0, 0.06, 40.0, 6.283185307179586, 1.0, -0.05, 0.0, 'canard', True), (5.0, 1.0, 0.06, 40.0, 6.283185307179586, 1.0, -0.05, 0.0, 'canard', False), (5.0, 1.0, 0.06, 40.0, 6.283185307179586, 1.0, 0.05, 0.0, 'tail', True), (5.0, 1.0, 0.06,

In [36]:
wing_area = span_max**2/max(lists_to_recombine['aspect_ratio'])

planforms:list[Planform] = list()

for planform_param in planform_params:
    span = np.sqrt(wing_area * planform_param['aspect_ratio'])

    planforms.append(Planform(
        aspect_ratio=planform_param['aspect_ratio'],
        taper=planform_param['taper'],
        sweep_quarter_deg=planform_param['sweep'],
        thickness_to_chord=planform_param['thickness_to_chord'],
        cm_quarter_chord=planform_param['cm_ac'],
        cl0=planform_param['cl_0'],
        clmax=planform_param['cl_max'],
        flap=False, #NOTE for now
        airfoil_lift_slope=planform_param['cl_alpha'],
        wetted_surface_ratio=1.07,
        interference_factor=1.,
        span=span
    ))

# Caching Planform properties

## CD0

In [ ]:
for planform in planforms:
    planform.add_cache_entry()